<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드, 저자: <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 효율적인 멀티헤드 어텐션(Multi-Head Attention) 구현 비교

이 코드 노트북은 GPT, Llama 등과 같은 디코더 스타일 LLM에서 사용되는 인과적 멀티헤드 어텐션(causal multi-head attention)을 구현하는 다양한 방법을 비교합니다.

In [ ]:
import torch

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch 버전: {torch.__version__}")

batch_size = 8
context_len = 1024
embed_dim = 768
embeddings = torch.randn((batch_size, context_len, embed_dim), device=device)

- 이 노트북의 모든 코드를 실행하려면 최소한 PyTorch 2.5로 업데이트했는지 확인하세요 (FlexAttention은 이전 PyTorch 버전에 포함되지 않음)
- 위 코드 셀이 2.5보다 낮은 PyTorch 버전을 표시하는 경우, 다음 코드 셀의 주석을 해제하고 실행하여 PyTorch 설치를 업그레이드할 수 있습니다 (PyTorch 2.5는 Python 3.9 이상이 필요함에 주의하세요)
- 더 구체적인 지침과 CUDA 버전에 대해서는 https://pytorch.org의 공식 설치 가이드를 참조하세요

In [ ]:
# pip install --upgrade torch torchvision torchaudio

<br>
&nbsp;

## 1) 3장의 CausalAttention MHA 래퍼 클래스

In [ ]:
import torch.nn as nn

class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)  # 새로 추가
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))  # 새로 추가

    def forward(self, x):
        b, num_tokens, d_in = x.shape  # 새로운 배치 차원 b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)  # transpose 변경
        attn_scores.masked_fill_(  # 새로 추가, _ ops는 in-place 연산
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)  # 새로 추가

        context_vec = attn_weights @ values
        return context_vec


class Ch03_MHA_Wrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
        self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)

    def forward(self, x):
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)


mha_ch03_wrapper = Ch03_MHA_Wrapper(
    d_in=embed_dim,
    d_out=embed_dim//12,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_ch03_wrapper(embeddings)
print(out.shape)

<br>
&nbsp;

## 2) 3장의 멀티헤드 어텐션(multi-head attention) 클래스

In [ ]:
class Ch03_MHA(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 원하는 출력 차원과 맞추기 위해 프로젝션 차원을 축소

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # 헤드 출력을 결합하는 선형 레이어
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # 형태: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # `num_heads` 차원을 추가하여 암시적으로 행렬을 분할
        # 마지막 차원을 펼침: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 전치: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 인과적 마스크를 사용한 스케일드 닷-프로덕트 어텐션(셀프 어텐션) 계산
        attn_scores = queries @ keys.transpose(2, 3)  # 각 헤드에 대한 닷 프로덕트

        # 토큰 수만큼 원본 마스크를 자르고 불리언으로 변환
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 마스크를 사용하여 어텐션 스코어를 채움
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 형태: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # 헤드들을 결합, 여기서 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # 선택적 프로젝션

        return context_vec


mha_ch03 = Ch03_MHA(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_ch03(embeddings)
print(out.shape)

<br>
&nbsp;

## 3) 결합된 가중치를 사용한 대안적 멀티헤드 어텐션

- 아래 `MultiHeadAttentionCombinedQKV` 클래스의 코드는 [Rayed Bin Wahed](https://github.com/rasbt/LLMs-from-scratch/discussions/51)가 친절하게 공유한 코드를 기반으로 합니다
- `MultiHeadAttentionCombinedQKV` 클래스와 3장에서 사용한 `MultiHeadAttention` 클래스의 주요 차이점은 `MultiHeadAttentionCombinedQKV`가 단일 가중치 행렬 `self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)`를 사용한다는 점입니다. 별도의 가중치 행렬 대신:

  - `self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)`
  - `self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)`
  - `self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)`

- 여기서 `self.qkv`는 세 개의 가중치 행렬 `self.W_query`, `self.W_key`, `self.W_value`를 모두 결합하여 쿼리, 키, 값 계산을 단일 단계로 수행합니다
- `q, k, v = qkv.unbind(0)`를 사용하여 개별 쿼리, 키, 값 텐서를 얻으며, 이들은 3장의 `MultiHeadAttention` 클래스에서와 유사하게 사용됩니다

In [ ]:
import torch.nn as nn


class MultiHeadAttentionCombinedQKV(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어떨어져야 합니다"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3개 (b, num_head, num_tokens, head_dim)
        queries, keys, values = qkv.unbind(0)

        # (b, num_heads, num_tokens, head_dim) --> (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(-2, -1)
        attn_scores = attn_scores.masked_fill(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**-0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # (b, num_heads, num_tokens, num_tokens) --> (b, num_heads, num_tokens, head_dim)
        context_vec = attn_weights @ values

        # (b, num_heads, num_tokens, head_dim) --> (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # (b, num_tokens, num_heads, head_dim) --> (b, num_tokens, embed_dim)
        context_vec = context_vec.contiguous().view(batch_size, num_tokens, embed_dim)

        context_vec = self.proj(context_vec)

        return context_vec


mha_combined_qkv = MultiHeadAttentionCombinedQKV(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_combined_qkv(embeddings)
print(out.shape)

<br>
&nbsp;

## 4) 아인슈타인 합(Einsum)을 사용한 멀티헤드 어텐션

- [`torch.einsum`](https://pytorch.org/docs/stable/generated/torch.einsum.html)을 통해 아인슈타인 합을 사용하여 멀티헤드 어텐션을 구현

In [ ]:
import math


class MHAEinsum(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Q, K, V를 위한 매개변수 초기화
        self.W_query = nn.Parameter(torch.randn(d_out, d_in))
        self.W_key = nn.Parameter(torch.randn(d_out, d_in))
        self.W_value = nn.Parameter(torch.randn(d_out, d_in))

        if qkv_bias:
            self.bias_q = nn.Parameter(torch.zeros(d_out))
            self.bias_k = nn.Parameter(torch.zeros(d_out))
            self.bias_v = nn.Parameter(torch.zeros(d_out))
        else:
            self.register_parameter("bias_q", None)
            self.register_parameter("bias_k", None)
            self.register_parameter("bias_v", None)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

        # 매개변수 초기화
        self.reset_parameters()


    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.W_query, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W_key, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W_value, a=math.sqrt(5))
        if self.bias_q is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.W_query)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias_q, -bound, bound)
            nn.init.uniform_(self.bias_k, -bound, bound)
            nn.init.uniform_(self.bias_v, -bound, bound)

    def forward(self, x):
        b, n, _ = x.shape

        # einsum을 사용하여 Q, K, V 계산, 먼저 선형 변환 수행
        Q = torch.einsum("bnd,di->bni", x, self.W_query)
        K = torch.einsum("bnd,di->bni", x, self.W_key)
        V = torch.einsum("bnd,di->bni", x, self.W_value)

        # 사용하는 경우 편향 추가
        if self.bias_q is not None:
            Q += self.bias_q
            K += self.bias_k
            V += self.bias_v

        # 멀티헤드 어텐션을 위한 형태 변경
        Q = Q.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

        # 스케일드 닷-프로덕트 어텐션
        scores = torch.einsum("bhnd,bhmd->bhnm", Q, K) / (self.head_dim ** 0.5)

        # 마스크 적용
        mask = self.mask[:n, :n].unsqueeze(0).unsqueeze(1).expand(b, self.num_heads, n, n)
        scores = scores.masked_fill(mask.bool(), -torch.inf)

        # 소프트맥스와 드롭아웃
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 어텐션된 컨텍스트 벡터들을 집계
        context_vec = torch.einsum("bhnm,bhmd->bhnd", attn_weights, V)

        # 헤드들을 결합하고 출력을 프로젝션
        context_vec = context_vec.transpose(1, 2).reshape(b, n, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec


mha_einsum = MHAEinsum(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_einsum(embeddings)
print(out.shape)

<br>
&nbsp;

## 5) PyTorch의 스케일드 닷 프로덕트 어텐션(scaled dot product attention)과 FlashAttention을 사용한 멀티헤드 어텐션

- 아래 구현은 [FlashAttention](https://arxiv.org/abs/2205.14135)이라는 메모리 최적화된 셀프 어텐션 버전을 구현하는 PyTorch의 [`scaled_dot_product_attention`](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) 함수를 사용합니다

In [ ]:
class MHAPyTorchScaledDotProduct(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어떨어져야 합니다"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads
        self.d_out = d_out

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3개 (b, num_heads, num_tokens, head_dim)
        queries, keys, values = qkv

        use_dropout = 0. if not self.training else self.dropout

        context_vec = nn.functional.scaled_dot_product_attention(
            queries, keys, values, attn_mask=None, dropout_p=use_dropout, is_causal=True)

        # 헤드들을 결합, 여기서 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)

        context_vec = self.proj(context_vec)

        return context_vec

In [ ]:
mha_pytorch_scaled = MHAPyTorchScaledDotProduct(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_pytorch_scaled(embeddings)
print(out.shape)

<br>
&nbsp;

## 6) FlashAttention 없는 PyTorch의 스케일드 닷 프로덕트 어텐션

- 위와 유사하지만 명시적인 인과적 마스크를 전달하여 FlashAttention을 비활성화합니다

In [ ]:
class MHAPyTorchSDPAWithoutFlash(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어떨어져야 합니다"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads
        self.d_out = d_out

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3개 (b, num_heads, num_tokens, head_dim)
        queries, keys, values = qkv

        use_dropout = 0. if not self.training else self.dropout

        # attn_mask가 예상 형태와 호환되고 `batch_first=True`와 맞는지 확인
        # num_heads에 대해 수동으로 조정할 필요 없음; 시퀀스에 대해 올바른지 확인
        if self.context_length >= num_tokens:
            attn_mask = self.mask[:num_tokens, :num_tokens]
        else:
            attn_mask = self.mask[:self.context_length, :self.context_length]

        context_vec = nn.functional.scaled_dot_product_attention(
            queries, keys, values, attn_mask=attn_mask, dropout_p=use_dropout, is_causal=False)

        # 헤드들을 결합, 여기서 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)

        context_vec = self.proj(context_vec)

        return context_vec

In [ ]:
mha_pytorch_sdpa_no_flash = MHAPyTorchSDPAWithoutFlash(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_pytorch_sdpa_no_flash(embeddings)
print(out.shape)

<br>
&nbsp;

## 7) PyTorch의 torch.nn.MultiheadAttention 사용

- 아래에서는 PyTorch의 [torch.nn.MultiheadAttention](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) 구현을 사용합니다

In [ ]:
import torch.nn as nn


class MHAPyTorchClass(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False, need_weights=True):
        super().__init__()

        self.context_length = context_length
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=d_out,
            num_heads=num_heads,
            dropout=dropout,
            bias=qkv_bias,
            add_bias_kv=qkv_bias,
            batch_first=True,
        )

        self.need_weights = need_weights
        self.proj = nn.Linear(d_out, d_out)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    def forward(self, x):
        batch_size, num_tokens, _ = x.shape

        # attn_mask가 예상 형태와 호환되고 `batch_first=True`와 맞는지 확인
        # num_heads에 대해 수동으로 조정할 필요 없음; 시퀀스에 대해 올바른지 확인
        if self.context_length >= num_tokens:
            attn_mask = self.mask[:num_tokens, :num_tokens]
        else:
            attn_mask = self.mask[:self.context_length, :self.context_length]

        # attn_mask 브로드캐스팅은 batch_size 차원을 암묵적으로 처리
        attn_output, _ = self.multihead_attn(
            x, x, x, attn_mask=attn_mask, need_weights=self.need_weights
        )

        output = self.proj(attn_output)

        return output


mha_pytorch_class_default = MHAPyTorchClass(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_pytorch_class_default(embeddings)
print(out.shape)

<br>
&nbsp;

## 8) `scaled_dot_product_attention`과 함께 PyTorch의 torch.nn.MultiheadAttention 사용

- `MultiheadAttention`이 [문서에 따라](https://github.com/pytorch/pytorch/blob/71d020262793542974cf13b30f2a9099773f015c/torch/nn/modules/activation.py#L1096) `scaled_dot_product_attention`을 사용하도록 `need_weights` (기본값 `True`)를 `False`로 설정합니다

```markdown
need_weights: 지정된 경우 `attn_outputs` 외에 `attn_output_weights`를 반환합니다.
           최적화된 `scaled_dot_product_attention`을 사용하고
           MHA에서 최고 성능을 달성하려면 `need_weights=False`로 설정하세요.
           기본값: `True`
```

In [ ]:
mha_pytorch_class_noweights = MHAPyTorchClass(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False,
    need_weights=False # 새로 추가!
).to(device)

out = mha_pytorch_class_noweights(embeddings)
print(out.shape)

<br>
&nbsp;

## 9) PyTorch의 FlexAttention 사용

- FlexAttention에 대해 더 알아보려면 [FlexAttention: The Flexibility of PyTorch with the Performance of FlashAttention](https://pytorch.org/blog/flexattention/)을 참조하세요
- FlexAttention 주의사항: 현재 드롭아웃을 지원하지 않습니다
- PyTorch 2.5부터 지원되며, CPU 머신에서는 다음을 통해 설치할 수 있습니다

    ```bash
    pip install torch torchvision torchaudio
    ```

- GPU 머신에 PyTorch를 설치하려면 다음을 사용하세요 (자세한 정보는 [pytorch.org](https://pytorch.org/)의 설치 메뉴도 참조)

    ```bash
    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    ```

In [ ]:
from packaging.version import parse as parse_version

def normalize_version(version):
    parsed_version = parse_version(version)
    return parse_version(f"{parsed_version.major}.{parsed_version.minor}.{parsed_version.micro}")

current_version = normalize_version(torch.__version__)
MIN_TORCH_VERSION = "2.5.0"
required_version = parse_version(MIN_TORCH_VERSION)

In [ ]:
if current_version >= required_version and torch.cuda.is_available():
    from torch.nn.attention.flex_attention import flex_attention, create_block_mask


def causal(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx


class MHAPyTorchFlexAttention(nn.Module):

    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어떨어져야 합니다"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads
        self.d_out = d_out

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout
        # self.register_buffer("block_mask", create_block_mask(causal, B=None, H=None, Q_LEN=context_length, KV_LEN=context_length))
        # `create_block_mask` 함수는 아직 버퍼를 지원하지 않습니다
        self.block_mask = create_block_mask(causal, B=None, H=None, Q_LEN=context_length, KV_LEN=context_length)


    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3개 (b, num_heads, num_tokens, head_dim)
        queries, keys, values = qkv

        # use_dropout = 0. if not self.training else self.dropout

        # attn_mask가 예상 형태와 호환되고 `batch_first=True`와 맞는지 확인
        # num_heads에 대해 수동으로 조정할 필요 없음; 시퀀스에 대해 올바른지 확인
        if self.context_length >= num_tokens:
            attn_mask = self.block_mask[:num_tokens, :num_tokens]
        else:
            attn_mask = self.block_mask[:self.context_length, :self.context_length]

        context_vec = flex_attention(queries, keys, values, block_mask=attn_mask)

        # 헤드들을 결합, 여기서 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)

        context_vec = self.proj(context_vec)

        return context_vec

In [ ]:
if current_version >= required_version and torch.cuda.is_available():

    mha_pytorch_flex = MHAPyTorchFlexAttention(
        d_in=embed_dim,
        d_out=embed_dim,
        context_length=context_len,
        dropout=0.0,
        num_heads=12,
        qkv_bias=False
    ).to(device)

    out = mha_pytorch_flex(embeddings)
    print(out.shape)

<br>
&nbsp;

## 빠른 속도 비교 (M3 Macbook Air CPU)

In [ ]:
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch 버전: {torch.__version__}")
print(f"{device}에서 실행 중")

In [ ]:
## 1) 3장의 CausalAttention MHA 래퍼 클래스
%timeit mha_ch03_wrapper(embeddings)

In [ ]:
## 2) 3장의 멀티헤드 어텐션 클래스
%timeit mha_ch03(embeddings)

In [ ]:
## 3) 결합된 가중치를 사용한 대안적 멀티헤드 어텐션
%timeit mha_combined_qkv(embeddings)

In [ ]:
## 4) 아인슈타인 합을 사용한 멀티헤드 어텐션
%timeit mha_einsum(embeddings)

In [ ]:
## 5) PyTorch의 스케일드 닷 프로덕트 어텐션을 사용한 멀티헤드 어텐션
%timeit mha_pytorch_scaled(embeddings)

In [ ]:
## 6) FlashAttention 없는 PyTorch의 스케일드 닷 프로덕트 어텐션
%timeit mha_pytorch_sdpa_no_flash(embeddings)

In [ ]:
## 7) PyTorch의 torch.nn.MultiheadAttention 사용
%timeit mha_pytorch_class_default(embeddings)

In [ ]:
## 8) `need_weights`를 비활성화한 PyTorch의 torch.nn.MultiheadAttention 사용
%timeit mha_pytorch_class_noweights(embeddings)

In [ ]:
## 9) PyTorch의 FlexAttention 사용

# PyTorch 2.5.0 이상이 필요하며 현재 CUDA PyTorch만 지원
%timeit mha_pytorch_flex(embeddings)

<br>
&nbsp;

## 빠른 속도 비교 (Nvidia A100 GPU)

In [ ]:
# 텐서 코어 활성화
torch.set_float32_matmul_precision("high")

In [ ]:
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch 버전: {torch.__version__}")
print(f"{device}에서 실행 중")

In [ ]:
## 1) 3장의 CausalAttention MHA 래퍼 클래스
%timeit mha_ch03_wrapper(embeddings)

In [ ]:
## 2) 3장의 멀티헤드 어텐션 클래스
%timeit mha_ch03(embeddings)

In [ ]:
## 3) 결합된 가중치를 사용한 대안적 멀티헤드 어텐션
%timeit mha_combined_qkv(embeddings)

In [ ]:
## 4) 아인슈타인 합을 사용한 멀티헤드 어텐션
%timeit mha_einsum(embeddings)

In [ ]:
## 5) PyTorch의 스케일드 닷 프로덕트 어텐션을 사용한 멀티헤드 어텐션
%timeit mha_pytorch_scaled(embeddings)

In [ ]:
## 6) FlashAttention 없는 PyTorch의 스케일드 닷 프로덕트 어텐션
%timeit mha_pytorch_sdpa_no_flash(embeddings)

In [ ]:
## 7) PyTorch의 torch.nn.MultiheadAttention 사용
%timeit mha_pytorch_class_default(embeddings)

In [ ]:
## 8) `need_weights`를 비활성화한 PyTorch의 torch.nn.MultiheadAttention 사용
%timeit mha_pytorch_class_noweights(embeddings)

In [ ]:
## 9) PyTorch의 FlexAttention 사용

# PyTorch 2.5.0 이상이 필요
%timeit mha_pytorch_flex(embeddings)

<br>
&nbsp;


# 시각화

In [ ]:
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch 버전: {torch.__version__}")
print(f"{device}에서 실행 중")

In [ ]:
functions = {
    "1) MHA 래퍼 클래스": mha_ch03_wrapper,
    "2) MHA Ch03": mha_ch03,
    "3) 결합된 QKV 가중치를 가진 MHA": mha_combined_qkv,
    "4) Einsum을 사용한 MHA": mha_einsum,
    "5) PyTorch scaled_dot_product_attention을 사용한 MHA": mha_pytorch_scaled,
    "6) FlashAttention 없는 PyTorch SDPA": mha_pytorch_sdpa_no_flash,
    "7) PyTorch MHA 클래스 기본설정": mha_pytorch_class_default,
    "8) need_weights=False인 PyTorch MHA": mha_pytorch_class_noweights
    }

if current_version >= required_version and torch.cuda.is_available():
    functions["9) PyTorch의 FlexAttention"] =  mha_pytorch_flex

In [ ]:
import matplotlib.pyplot as plt

# 다크 모드 미학을 위한 추가 커스터마이징
plt.rcParams["figure.facecolor"] = "#121212"
plt.rcParams["axes.facecolor"] = "#121212"
plt.rcParams["axes.edgecolor"] = "white"
plt.rcParams["axes.labelcolor"] = "white"
plt.rcParams["text.color"] = "white"
plt.rcParams["xtick.color"] = "white"
plt.rcParams["ytick.color"] = "white"
plt.rcParams["grid.color"] = "#444444"
plt.rcParams["lines.linewidth"] = 2
plt.rcParams["lines.markersize"] = 8

def plot_execution_times(functions, execution_means, execution_stds, filename):

    # 플롯 생성
    fig, ax = plt.subplots()
    bars = ax.bar(functions.keys(), execution_means, yerr=execution_stds, capsize=5, error_kw={'ecolor': 'grey'})

    plt.ylabel("실행 시간 (ms)")
    plt.xticks(rotation=45, ha="right")

    # 여백을 가진 새로운 ylim 계산
    max_execution_time = max(execution_means)
    upper_ylim = max_execution_time + 0.4 * max_execution_time  # 40% 여백 추가
    plt.ylim(0, upper_ylim)

    # 막대에 실행 시간으로 주석 표시
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + (0.05 * upper_ylim), round(yval, 2), ha="center", va="bottom")

    plt.tight_layout()
    plt.savefig(filename)
    plt.show()

## 워밍업을 포함한 속도 비교 (Nvidia A100 GPU) (순전파만)

In [ ]:
# Andrei Aksionov가 공유한 CUDA 벤치마크 코드
# https://github.com/cuda-mode/lectures/blob/main/lecture1/pytorch_square.py 코드 기반

import numpy as np

def time_pytorch_function(func, *input, num_repeats=1_000):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    # 워밍업
    for _ in range(5):
        func(*input)
    torch.cuda.synchronize()

    times = []
    for _ in range(num_repeats):
        start.record()
        func(*input)
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))

    return np.mean(times), np.std(times)

In [ ]:
execution_stats = [time_pytorch_function(fn, embeddings) for fn in functions.values()]
execution_means = [stat[0] for stat in execution_stats]
execution_stds = [stat[1] for stat in execution_stats]


plot_execution_times(functions, execution_means, execution_stds, filename="1_forward-only.pdf")

<br>
&nbsp;


## 워밍업을 포함한 속도 비교 (Nvidia A100 GPU) (순전파 및 역전파)

In [ ]:
def forward_backward(func, embeddings):
    if embeddings.grad is not None:
        embeddings.grad.zero_()

    output = func(embeddings)
    loss = output.sum()
    loss.backward()


def time_pytorch_function_forward_backward(func, *input, num_repeats = 1_000):
    # CUDA는 비동기이므로 python time 모듈 사용 불가
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    # 워밍업
    for _ in range(5):
        forward_backward(func, *input)
    torch.cuda.synchronize()

    times = []
    for _ in range(num_repeats):
        start.record()
        forward_backward(func, *input)
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))

    return np.mean(times), np.std(times)

In [ ]:
execution_stats = [time_pytorch_function_forward_backward(fn, embeddings) for fn in functions.values()]
execution_means = [stat[0] for stat in execution_stats]
execution_stds = [stat[1] for stat in execution_stats]


plot_execution_times(functions, execution_means, execution_stds, filename="2_forward-and-backward.pdf")

<br>
&nbsp;


## 워밍업과 컴파일을 포함한 속도 비교 (Nvidia A100 GPU) (순전파 및 역전파)

In [ ]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True

def prepare_function(fn):
    fn = torch.compile(fn)
    return fn

In [ ]:
execution_stats = [time_pytorch_function_forward_backward(prepare_function(fn), embeddings) for fn in functions.values()]
execution_means = [stat[0] for stat in execution_stats]
execution_stds = [stat[1] for stat in execution_stats]


plot_execution_times(functions, execution_means, execution_stds, filename="3_forward-and-backward-compiled.pdf")